# FinNexus ML Training Notebook
## Comprehensive Machine Learning Models for FinNexus Fintech Platform

This notebook trains and saves all ML models needed for the FinNexus platform:
1. **Stock Direction Predictor** - For Playground bot predictions
2. **Volatility/Risk Scorer** - For Portfolio Analyzer risk scoring
3. **News Sentiment (FinBERT)** - For News Intelligence module
4. **Portfolio Similarity Scorer** - For AI Advisor portfolio classification

## Requirements
```
yfinance==0.2.28
pandas==2.0.3
numpy==1.24.3
pandas-ta==0.3.14b
scikit-learn==1.3.0
joblib==1.3.2
transformers==4.31.0
torch==2.0.1
matplotlib==3.7.2
seaborn==0.12.2
warnings
os
json
```

═══════════════════════════════════════════════
SECTION 1 — Setup & Imports
═══════════════════════════════════════════════

In [ ]:
# Install required packages
!pip install yfinance pandas pandas-ta scikit-learn joblib transformers torch matplotlib seaborn -q

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import json
import numpy as np
import pandas as pd
import pandas_ta as ta
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# ML Libraries
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report,
                             roc_curve, auc, roc_auc_score)
import joblib

print('✅ All imports successful!')

═══════════════════════════════════════════════
SECTION 2 — Data Collection
═══════════════════════════════════════════════

In [ ]:
# Define assets to download
STOCKS = ['AAPL', 'GOOGL', 'TSLA', 'MSFT', 'AMZN', 'RELIANCE.NS', 'TCS.NS']
CRYPTO = ['BTC-USD', 'ETH-USD', 'SOL-USD', 'BNB-USD']
COMMODITIES = ['GC=F', 'CL=F']  # Gold, Oil
INDICES = ['^NSEI', '^GSPC', '^IXIC']
FOREX = ['USDINR=X', 'EURUSD=X']

ALL_ASSETS = STOCKS + CRYPTO + COMMODITIES + INDICES + FOREX

# Date range
START_DATE = '2018-01-01'
END_DATE = '2023-12-31'

print(f'Downloading data for {len(ALL_ASSETS)} assets from {START_DATE} to {END_DATE}')

In [ ]:
# Download all data
data = {}

for symbol in ALL_ASSETS:
    try:
        print(f'Downloading {symbol}...', end=' ')
        df = yf.download(symbol, start=START_DATE, end=END_DATE, interval='1d', progress=False)
        if len(df) > 0:
            # Flatten multi-index columns if present
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)
            data[symbol] = df
            print(f'✅ {len(df)} rows')
        else:
            print('⚠️ No data')
    except Exception as e:
        print(f'❌ Error: {e}')

print(f'\n✅ Downloaded {len(data)} out of {len(ALL_ASSETS)} assets')

In [ ]:
# Print shape and head of each dataframe
for symbol, df in data.items():
    print(f'\n{symbol}: {df.shape}')
    print(df.head(2))

In [ ]:
# Plot normalized closing prices for all assets
plt.figure(figsize=(16, 8))

normalized_data = pd.DataFrame()
for symbol, df in data.items():
    if 'Close' in df.columns:
        normalized_data[symbol] = df['Close'] / df['Close'].iloc[0]

normalized_data.plot(ax=plt.gca(), title='Normalized Closing Prices (2018-2023)',
                     xlabel='Date', ylabel='Normalized Price')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

═══════════════════════════════════════════════
SECTION 3 — Feature Engineering
═══════════════════════════════════════════════

In [ ]:
def add_features(df, symbol):
    """Add technical indicators and features to a dataframe"""
    df = df.copy()

    # Ensure we have required columns
    required_cols = ['Open', 'High', 'Low', 'Close', 'Volume']
    for col in required_cols:
        if col not in df.columns:
            return None

    # Price Features
    df['prev_close'] = df['Close'].shift(1)
    df['daily_return'] = (df['Close'] - df['prev_close']) / df['prev_close']
    df['log_return'] = np.log(df['Close'] / df['prev_close'])

    # Moving Averages
    df['SMA20'] = ta.sma(df['Close'], length=20)
    df['SMA50'] = ta.sma(df['Close'], length=50)
    df['price_vs_20MA'] = (df['Close'] - df['SMA20']) / df['SMA20']
    df['price_vs_50MA'] = (df['Close'] - df['SMA50']) / df['SMA50']

    # High-Low range
    df['high_low_range'] = (df['High'] - df['Low']) / df['Close']

    # RSI
    df['RSI'] = ta.rsi(df['Close'], length=14)

    # MACD
    macd = ta.macd(df['Close'], fast=12, slow=26, signal=9)
    df['MACD'] = macd['MACD_12_26_9']
    df['MACD_signal'] = macd['MACDs_12_26_9']
    df['MACD_hist'] = macd['MACDh_12_26_9']

    # Bollinger Bands
    bbands = ta.bbands(df['Close'], length=20, std=2)
    df['BB_upper'] = bbands['BBU_20_2.0']
    df['BB_middle'] = bbands['BBM_20_2.0']
    df['BB_lower'] = bbands['BBL_20_2.0']
    df['BB_position'] = (df['Close'] - df['BB_lower']) / (df['BB_upper'] - df['BB_lower'])

    # EMA
    df['EMA12'] = ta.ema(df['Close'], length=12)
    df['EMA26'] = ta.ema(df['Close'], length=26)

    # ATR
    df['ATR'] = ta.atr(df['High'], df['Low'], df['Close'], length=14)

    # OBV
    df['OBV'] = ta.obv(df['Close'], df['Volume'])

    # Stochastic
    stoch = ta.stoch(df['High'], df['Low'], df['Close'], k=14, d=3)
    df['Stoch_K'] = stoch['STOCHk_14_3_3']
    df['Stoch_D'] = stoch['STOCHd_14_3_3']

    # ADX
    df['ADX'] = ta.adx(df['High'], df['Low'], df['Close'], length=14)['ADX_14']

    # Target Variable
    df['next_close'] = df['Close'].shift(-1)
    df['direction'] = (df['next_close'] > df['Close']).astype(int)
    df['magnitude'] = (df['next_close'] - df['Close']) / df['Close']

    df['symbol'] = symbol

    return df

print('✅ Feature engineering function defined')

In [ ]:
# Process all dataframes
processed_data = []

for symbol, df in data.items():
    print(f'Processing {symbol}...', end=' ')
    processed = add_features(df, symbol)
    if processed is not None:
        processed_data.append(processed)
        print(f'✅ {len(processed)} rows')
    else:
        print('❌ Failed')

# Combine all dataframes
master_df = pd.concat(processed_data, ignore_index=True)
print(f'\n✅ Master dataframe shape: {master_df.shape}')

In [ ]:
# Drop NaN rows
print(f'Before dropping NaN: {master_df.shape}')
master_df = master_df.dropna()
print(f'After dropping NaN: {master_df.shape}')

In [ ]:
# Feature correlation heatmap
feature_cols = ['daily_return', 'log_return', 'price_vs_20MA', 'price_vs_50MA',
                'high_low_range', 'RSI', 'MACD', 'MACD_signal', 'MACD_hist',
                'BB_position', 'EMA12', 'EMA26', 'ATR', 'OBV', 'Stoch_K',
                'Stoch_D', 'ADX']

plt.figure(figsize=(14, 10))
corr_matrix = master_df[feature_cols].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Class balance
print('Class Distribution:')
print(master_df['direction'].value_counts())
print(f"\nClass balance: {master_df['direction'].value_counts(normalize=True).to_dict()}")

═══════════════════════════════════════════════
SECTION 4 — Model 1: Stock Direction Predictor
═══════════════════════════════════════════════

In [ ]:
# Prepare features and target
FEATURE_COLS = ['daily_return', 'log_return', 'price_vs_20MA', 'price_vs_50MA',
                'high_low_range', 'RSI', 'MACD', 'MACD_signal', 'MACD_hist',
                'BB_position', 'EMA12', 'EMA26', 'ATR', 'OBV', 'Stoch_K',
                'Stoch_D', 'ADX']

X = master_df[FEATURE_COLS].copy()
y = master_df['direction'].copy()

# Train/test split - NO shuffle (time series)
split_idx = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f'Training set: {X_train.shape}')
print(f'Test set: {X_test.shape}')

In [ ]:
# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('✅ Features scaled')

In [ ]:
# Train models
models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42),
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000)
}

results = {}

for name, model in models.items():
    print(f'\nTraining {name}...')
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1] if hasattr(model, 'predict_proba') else None

    results[name] = {
        'model': model,
        'predictions': y_pred,
        'probabilities': y_prob,
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred),
        'roc_auc': roc_auc_score(y_test, y_prob) if y_prob is not None else None
    }

    print(f"  Accuracy:  {results[name]['accuracy']:.4f}")
    print(f"  Precision: {results[name]['precision']:.4f}")
    print(f"  Recall:    {results[name]['recall']:.4f}")
    print(f"  F1 Score:  {results[name]['f1']:.4f}")
    print(f"  ROC AUC:   {results[name]['roc_auc']:.4f}")

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (name, res) in enumerate(results.items()):
    cm = confusion_matrix(y_test, res['predictions'])
    sns.heatmap(cm, annot=True, fmt='d', ax=axes[idx], cmap='Blues')
    axes[idx].set_title(f"{name}\nAccuracy: {res['accuracy']:.4f}")
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('Actual')

plt.tight_layout()
plt.show()

In [ ]:
# ROC Curves
plt.figure(figsize=(10, 8))

for name, res in results.items():
    if res['probabilities'] is not None:
        fpr, tpr, _ = roc_curve(y_test, res['probabilities'])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f'{name} (AUC = {roc_auc:.4f})')

plt.plot([0, 1], [0, 1], 'k--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Select best model by F1 score
best_model_name = max(results, key=lambda x: results[x]['f1'])
best_model = results[best_model_name]['model']
best_f1 = results[best_model_name]['f1']
best_accuracy = results[best_model_name]['accuracy']

print(f'Best Model: {best_model_name}')
print(f'F1 Score:   {best_f1:.4f}')
print(f'Accuracy:   {best_accuracy:.4f}')

In [ ]:
# Feature importance
if hasattr(best_model, 'feature_importances_'):
    plt.figure(figsize=(12, 6))
    importance = pd.DataFrame({
        'feature': FEATURE_COLS,
        'importance': best_model.feature_importances_
    }).sort_values('importance', ascending=True)

    plt.barh(importance['feature'], importance['importance'])
    plt.xlabel('Importance')
    plt.ylabel('Feature')
    plt.title(f'Feature Importance - {best_model_name}')
    plt.tight_layout()
    plt.show()
else:
    print('Feature importance not available for this model type')

In [ ]:
# Create output directories
os.makedirs('models/direction_predictor', exist_ok=True)
os.makedirs('models/risk_scorer', exist_ok=True)
os.makedirs('models/portfolio_classifier', exist_ok=True)

# Save direction predictor models
joblib.dump(best_model, 'models/direction_predictor/best_direction_model.joblib')
joblib.dump(scaler, 'models/direction_predictor/direction_scaler.joblib')

with open('models/direction_predictor/feature_columns.json', 'w') as f:
    json.dump(FEATURE_COLS, f)

print('✅ Direction predictor models saved')

═══════════════════════════════════════════════
SECTION 5 — Model 2: Volatility / Risk Scorer
═══════════════════════════════════════════════

In [ ]:
# Calculate risk metrics for each asset
risk_data = []

for symbol, df in data.items():
    if len(df) < 90:
        continue

    df = df.copy()
    df['returns'] = df['Close'].pct_change()

    # 30-day rolling volatility (annualized)
    df['rolling_vol'] = df['returns'].rolling(30).std() * np.sqrt(252)

    # Max drawdown over 90-day rolling window
    def max_drawdown(prices):
        peak = prices.expanding(min_periods=1).max()
        drawdown = (prices - peak) / peak
        return drawdown.min()

    df['max_drawdown'] = df['Close'].rolling(90).apply(max_drawdown, raw=False)

    # Sharpe ratio (rolling 30-day)
    risk_free = 0.065
    df['rolling_sharpe'] = (
        (df['returns'].rolling(30).mean() - risk_free / 252)
        / df['returns'].rolling(30).std()
    )

    # Beta vs S&P 500 (rolling 60-day)
    if '^GSPC' in data and len(data['^GSPC']) > 0:
        sp500 = data['^GSPC']['Close'].pct_change()
        cov = df['returns'].rolling(60).cov(sp500)
        var = sp500.rolling(60).var()
        df['beta'] = cov / var
    else:
        df['beta'] = 1.0

    latest = df.dropna().iloc[-1] if len(df.dropna()) > 0 else None

    if latest is not None:
        risk_data.append({
            'symbol': symbol,
            'volatility': latest['rolling_vol'],
            'max_drawdown': latest['max_drawdown'],
            'sharpe_ratio': latest['rolling_sharpe'],
            'beta': latest['beta'] if pd.notna(latest['beta']) else 1.0
        })

risk_df = pd.DataFrame(risk_data)
print(f'Risk metrics calculated for {len(risk_df)} assets')
print(risk_df.head(10))

In [ ]:
# Create risk labels
def get_risk_label(volatility):
    if volatility < 0.15:
        return 0  # Low risk
    elif volatility <= 0.35:
        return 1  # Medium risk
    else:
        return 2  # High risk

risk_df['risk_label'] = risk_df['volatility'].apply(get_risk_label)

print('Risk Label Distribution:')
print(risk_df['risk_label'].value_counts())

In [ ]:
# Prepare features for risk scorer
risk_features = ['volatility', 'max_drawdown', 'sharpe_ratio', 'beta']

X_risk = risk_df[risk_features].copy()
y_risk = risk_df['risk_label'].copy()

X_risk_train, X_risk_test, y_risk_train, y_risk_test = train_test_split(
    X_risk, y_risk, test_size=0.2, random_state=42
)

risk_scaler = StandardScaler()
X_risk_train_scaled = risk_scaler.fit_transform(X_risk_train)
X_risk_test_scaled = risk_scaler.transform(X_risk_test)

print(f'Training set: {X_risk_train.shape}')
print(f'Test set:     {X_risk_test.shape}')

In [ ]:
# Train Risk Scorer
risk_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
risk_model.fit(X_risk_train_scaled, y_risk_train)

y_risk_pred = risk_model.predict(X_risk_test_scaled)

print('Classification Report:')
print(classification_report(y_risk_test, y_risk_pred,
                            target_names=['Low Risk', 'Medium Risk', 'High Risk']))

In [ ]:
# Confusion matrix
plt.figure(figsize=(8, 6))
cm_risk = confusion_matrix(y_risk_test, y_risk_pred)
sns.heatmap(cm_risk, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Low Risk', 'Medium Risk', 'High Risk'],
            yticklabels=['Low Risk', 'Medium Risk', 'High Risk'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Risk Scorer Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Example predictions
print('Example Predictions:')
for i in range(min(5, len(X_risk_test))):
    actual = y_risk_test.iloc[i]
    pred = y_risk_pred[i]
    vol = X_risk_test.iloc[i]['volatility']
    sym = risk_df.iloc[X_risk_test.index[i]]['symbol'] if X_risk_test.index[i] < len(risk_df) else 'N/A'
    print(f'  {sym:<15} Volatility: {vol:.4f}  Actual: {actual}  Predicted: {pred}')

In [ ]:
# Save risk scorer models
joblib.dump(risk_model, 'models/risk_scorer/risk_scorer_model.joblib')
joblib.dump(risk_scaler, 'models/risk_scorer/risk_scaler.joblib')

print('✅ Risk scorer models saved')

═══════════════════════════════════════════════
SECTION 6 — Model 3: News Sentiment (FinBERT)
═══════════════════════════════════════════════

In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification
import torch

# Load FinBERT
print('Loading FinBERT model...')
tokenizer = BertTokenizer.from_pretrained('ProsusAI/finbert')
finbert_model = BertForSequenceClassification.from_pretrained('ProsusAI/finbert')
finbert_model.eval()

print('✅ FinBERT loaded successfully')

In [ ]:
def test_finbert(headlines):
    """Analyze sentiment of financial headlines using FinBERT"""
    results = []
    labels = ['negative', 'neutral', 'positive']

    for headline in headlines:
        inputs = tokenizer(headline, return_tensors='pt', truncation=True, max_length=512)

        with torch.no_grad():
            outputs = finbert_model(**inputs)
            probs = torch.softmax(outputs.logits, dim=1)
            pred = torch.argmax(probs, dim=1).item()
            confidence = probs[0][pred].item()

        results.append({
            'headline': headline,
            'sentiment': labels[pred],
            'confidence': confidence
        })

    return results


# Sample financial headlines
sample_headlines = [
    'Federal Reserve raises interest rates by 25 basis points',
    'Apple reports earnings beat, revenue up 15% year-over-year',
    'Bitcoin crashes 20% as crypto markets face massive selloff',
    'Gold surges to all-time high amid inflation fears',
    'Economists warn of potential recession in Q4',
    'Tech stocks rally as markets reach new record highs',
    'Oil prices drop amid weakening global demand',
    'Microsoft announces strategic partnership with AI startup',
    'Banking sector faces headwinds from credit losses',
    'Retail sales exceed expectations, consumer spending strong'
]

print('Testing FinBERT with sample headlines...')

In [ ]:
import time

# Run sentiment analysis
start_time = time.time()
sentiment_results = test_finbert(sample_headlines)
total_time = time.time() - start_time

# Print results
print('\n' + '=' * 80)
print(f"{'Headline':<50} | {'Sentiment':<10} | {'Confidence':<10}")
print('=' * 80)
for result in sentiment_results:
    print(f"{result['headline'][:48]:<50} | {result['sentiment']:<10} | {result['confidence']:.4f}")

print(f'\nModel size: ~110M parameters')
print(f'Total inference time: {total_time:.2f}s')
print(f'Average time per headline: {total_time / len(sample_headlines):.3f}s')

In [ ]:
# Sentiment distribution plot
sentiment_counts = pd.Series([r['sentiment'] for r in sentiment_results]).value_counts()

colors = {'positive': 'green', 'neutral': 'gray', 'negative': 'red'}
bar_colors = [colors.get(s, 'blue') for s in sentiment_counts.index]

plt.figure(figsize=(10, 6))
sentiment_counts.plot(kind='bar', color=bar_colors)
plt.title('FinBERT Sentiment Distribution')
plt.xlabel('Sentiment')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

> **Note:** FinBERT loads from HuggingFace at runtime — no local save needed. Use `ProsusAI/finbert` directly in your FastAPI service.

═══════════════════════════════════════════════
SECTION 7 — Model 4: Portfolio Similarity Scorer
═══════════════════════════════════════════════

In [ ]:
# Generate synthetic portfolio dataset
np.random.seed(42)
num_portfolios = 1000

asset_classes = ['stocks', 'crypto', 'gold', 'bonds_proxy', 'cash']
typical_returns = {'stocks': 0.10, 'crypto': 0.25, 'gold': 0.05, 'bonds_proxy': 0.03, 'cash': 0.02}

portfolios = []
for _ in range(num_portfolios):
    alloc = np.random.random(len(asset_classes))
    alloc = alloc / alloc.sum()

    portfolio = dict(zip(asset_classes, alloc))

    # Expected return
    portfolio['expected_return'] = sum(portfolio[k] * typical_returns[k] for k in asset_classes)

    # Portfolio volatility
    portfolio['volatility'] = sum(portfolio[k] * np.random.uniform(0.05, 0.40) for k in asset_classes)

    # Sharpe ratio
    portfolio['sharpe_ratio'] = (
        (portfolio['expected_return'] - 0.065) / max(portfolio['volatility'], 0.01)
    )

    # Diversification score (1 - HHI)
    hhi = sum(portfolio[k] ** 2 for k in asset_classes)
    portfolio['diversification_score'] = 1 - hhi

    portfolios.append(portfolio)

portfolio_df = pd.DataFrame(portfolios)
print(f'Generated {len(portfolio_df)} synthetic portfolios')
print(portfolio_df.head())

In [ ]:
# Label portfolios
def label_portfolio(row):
    if row['crypto'] > 0.30 or max(row[k] for k in asset_classes) > 0.60:
        return 'aggressive'
    elif row['diversification_score'] > 0.70 and row['sharpe_ratio'] > 1.0:
        return 'balanced'
    elif (row['stocks'] + row['gold'] > 0.70) and row['crypto'] < 0.10:
        return 'conservative'
    else:
        return 'risky'

portfolio_df['label'] = portfolio_df.apply(label_portfolio, axis=1)

print('Portfolio Label Distribution:')
print(portfolio_df['label'].value_counts())

In [ ]:
# Prepare features
portfolio_features = asset_classes + ['expected_return', 'volatility', 'sharpe_ratio', 'diversification_score']

X_port = portfolio_df[portfolio_features].copy()
y_port = portfolio_df['label'].copy()

X_port_train, X_port_test, y_port_train, y_port_test = train_test_split(
    X_port, y_port, test_size=0.2, random_state=42
)

portfolio_scaler = StandardScaler()
X_port_train_scaled = portfolio_scaler.fit_transform(X_port_train)
X_port_test_scaled = portfolio_scaler.transform(X_port_test)

print(f'Training set: {X_port_train.shape}')
print(f'Test set:     {X_port_test.shape}')

In [ ]:
# Train Portfolio Classifier
portfolio_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
portfolio_model.fit(X_port_train_scaled, y_port_train)

y_port_pred = portfolio_model.predict(X_port_test_scaled)

print('Classification Report:')
print(classification_report(y_port_test, y_port_pred))

In [ ]:
# Save portfolio classifier models
joblib.dump(portfolio_model, 'models/portfolio_classifier/portfolio_classifier.joblib')
joblib.dump(portfolio_scaler, 'models/portfolio_classifier/portfolio_scaler.joblib')

print('✅ Portfolio classifier models saved')

═══════════════════════════════════════════════
SECTION 8 — Save All Models + Create Summary
═══════════════════════════════════════════════

In [ ]:
# Create models summary
models_summary = {
    'direction_predictor': {
        'best_model': best_model_name,
        'accuracy': round(best_accuracy, 4),
        'f1_score': round(best_f1, 4),
        'features': FEATURE_COLS,
        'trained_on': '2018-2023',
        'assets': ALL_ASSETS
    },
    'risk_scorer': {
        'model': 'RandomForestClassifier',
        'accuracy': round(accuracy_score(y_risk_test, y_risk_pred), 4),
        'features': risk_features,
        'trained_on': '2018-2023',
        'classes': ['Low Risk', 'Medium Risk', 'High Risk']
    },
    'portfolio_classifier': {
        'model': 'RandomForestClassifier',
        'accuracy': round(accuracy_score(y_port_test, y_port_pred), 4),
        'features': portfolio_features,
        'classes': ['aggressive', 'balanced', 'conservative', 'risky']
    },
    'finbert': {
        'model': 'ProsusAI/finbert',
        'load_from': 'huggingface',
        'local_save': False,
        'num_labels': 3,
        'labels': ['negative', 'neutral', 'positive']
    }
}

with open('models/models_summary.json', 'w') as f:
    json.dump(models_summary, f, indent=2)

print('✅ Models summary saved')

In [ ]:
# Print final summary table
def get_file_size(filepath):
    if os.path.exists(filepath):
        return os.path.getsize(filepath) / 1024
    return 0

print('\n' + '=' * 100)
print(f"{'Model':<25} {'Purpose':<35} {'Accuracy':<12} {'File Size (KB)':<15}")
print('=' * 100)

rows = [
    ('Direction Predictor', 'Playground predictions',
     models_summary['direction_predictor']['accuracy'],
     get_file_size('models/direction_predictor/best_direction_model.joblib')),
    ('Risk Scorer', 'Portfolio risk scoring',
     models_summary['risk_scorer']['accuracy'],
     get_file_size('models/risk_scorer/risk_scorer_model.joblib')),
    ('Portfolio Classifier', 'AI Advisor comparison',
     models_summary['portfolio_classifier']['accuracy'],
     get_file_size('models/portfolio_classifier/portfolio_classifier.joblib')),
    ('FinBERT', 'News sentiment', None, None),
]

for name, purpose, acc, size in rows:
    acc_str = f'{acc:.4f}' if acc is not None else 'N/A (HuggingFace)'
    size_str = f'{size:.2f}' if size is not None else 'N/A'
    print(f'{name:<25} {purpose:<35} {acc_str:<12} {size_str:<15}')

print('=' * 100)

In [ ]:
# List all saved files
print('\n📁 Saved Model Files:')
for root, dirs, files in os.walk('models'):
    for file in files:
        filepath = os.path.join(root, file)
        size = os.path.getsize(filepath) / 1024
        print(f'  {filepath}: {size:.2f} KB')

═══════════════════════════════════════════════
SECTION 9 — FastAPI Integration Helper
═══════════════════════════════════════════════

> **Note:** Copy `ml_loader.py` into `backend/services/ml_service.py` after running this notebook.

In [ ]:
ml_loader_code = '''
import joblib
import json
import numpy as np
import pandas as pd
import torch
from transformers import BertTokenizer, BertForSequenceClassification
from datetime import datetime

# ── Global model holders ──────────────────────────────────────────────────────
_direction_model = None
_direction_scaler = None
_direction_features = None
_risk_model = None
_risk_scaler = None
_portfolio_model = None
_portfolio_scaler = None
_finbert_tokenizer = None
_finbert_model = None

PORTFOLIO_FEATURES = ['stocks', 'crypto', 'gold', 'bonds_proxy', 'cash',
                      'expected_return', 'volatility', 'sharpe_ratio', 'diversification_score']

TYPICAL_RETURNS = {'stocks': 0.10, 'crypto': 0.25, 'gold': 0.05, 'bonds_proxy': 0.03, 'cash': 0.02}


def load_all_models():
    global _direction_model, _direction_scaler, _direction_features
    global _risk_model, _risk_scaler
    global _portfolio_model, _portfolio_scaler
    global _finbert_tokenizer, _finbert_model

    print("[ML] Loading all models...")

    try:
        _direction_model = joblib.load("models/direction_predictor/best_direction_model.joblib")
        _direction_scaler = joblib.load("models/direction_predictor/direction_scaler.joblib")
        with open("models/direction_predictor/feature_columns.json") as f:
            _direction_features = json.load(f)
        print("[ML] ✅ Direction predictor loaded")
    except Exception as e:
        print(f"[ML] ❌ Direction predictor failed: {e}")

    try:
        _risk_model = joblib.load("models/risk_scorer/risk_scorer_model.joblib")
        _risk_scaler = joblib.load("models/risk_scorer/risk_scaler.joblib")
        print("[ML] ✅ Risk scorer loaded")
    except Exception as e:
        print(f"[ML] ❌ Risk scorer failed: {e}")

    try:
        _portfolio_model = joblib.load("models/portfolio_classifier/portfolio_classifier.joblib")
        _portfolio_scaler = joblib.load("models/portfolio_classifier/portfolio_scaler.joblib")
        print("[ML] ✅ Portfolio classifier loaded")
    except Exception as e:
        print(f"[ML] ❌ Portfolio classifier failed: {e}")

    try:
        _finbert_tokenizer = BertTokenizer.from_pretrained("ProsusAI/finbert")
        _finbert_model = BertForSequenceClassification.from_pretrained("ProsusAI/finbert")
        _finbert_model.eval()
        print("[ML] ✅ FinBERT loaded")
    except Exception as e:
        print(f"[ML] ❌ FinBERT failed: {e}")

    print("[ML] All models loaded.")


def predict_direction(ohlcv_df: pd.DataFrame) -> dict:
    """Predict price direction from OHLCV dataframe."""
    try:
        if _direction_model is None:
            return {"direction": "up", "confidence": 0.5, "fallback": True}

        features = ohlcv_df[_direction_features].iloc[[-1]].fillna(0)
        scaled = _direction_scaler.transform(features)
        pred = _direction_model.predict(scaled)[0]
        prob = _direction_model.predict_proba(scaled)[0][pred]

        return {
            "direction": "up" if pred == 1 else "down",
            "confidence": round(float(prob), 4),
            "timestamp": datetime.utcnow().isoformat()
        }
    except Exception as e:
        print(f"[ML] predict_direction error: {e}")
        return {"direction": "up", "confidence": 0.5, "fallback": True}


def score_portfolio_risk(portfolio_dict: dict) -> dict:
    """Score portfolio risk from volatility, drawdown, sharpe, beta."""
    try:
        if _risk_model is None:
            return {"risk_label": "medium", "score": 5, "fallback": True}

        features = pd.DataFrame([portfolio_dict])[['volatility', 'max_drawdown', 'sharpe_ratio', 'beta']].fillna(0)
        scaled = _risk_scaler.transform(features)
        pred = _risk_model.predict(scaled)[0]
        label_map = {0: "low", 1: "medium", 2: "high"}
        score_map = {0: 3, 1: 5, 2: 8}

        return {
            "risk_label": label_map[pred],
            "score": score_map[pred],
            "timestamp": datetime.utcnow().isoformat()
        }
    except Exception as e:
        print(f"[ML] score_portfolio_risk error: {e}")
        return {"risk_label": "medium", "score": 5, "fallback": True}


def classify_portfolio(allocations: dict) -> dict:
    """Classify portfolio type from asset allocations."""
    try:
        if _portfolio_model is None:
            return {"type": "balanced", "suggestions": [], "fallback": True}

        exp_return = sum(allocations.get(k, 0) * TYPICAL_RETURNS.get(k, 0.05) for k in TYPICAL_RETURNS)
        volatility = sum(allocations.get(k, 0) * 0.20 for k in TYPICAL_RETURNS)  # simplified
        sharpe = (exp_return - 0.065) / max(volatility, 0.01)
        hhi = sum(allocations.get(k, 0) ** 2 for k in TYPICAL_RETURNS)
        div_score = 1 - hhi

        row = {**{k: allocations.get(k, 0) for k in ['stocks', 'crypto', 'gold', 'bonds_proxy', 'cash']},
               'expected_return': exp_return, 'volatility': volatility,
               'sharpe_ratio': sharpe, 'diversification_score': div_score}

        features = pd.DataFrame([row])[PORTFOLIO_FEATURES].fillna(0)
        scaled = _portfolio_scaler.transform(features)
        pred = _portfolio_model.predict(scaled)[0]

        suggestions_map = {
            'aggressive': ['Consider reducing crypto exposure', 'Diversify into stable assets'],
            'balanced':   ['Portfolio looks healthy', 'Review quarterly for rebalancing'],
            'conservative': ['Consider small crypto allocation for growth', 'Monitor inflation impact on bonds'],
            'risky':      ['Improve diversification', 'Add gold as hedge', 'Reduce single-asset concentration']
        }

        return {
            "type": pred,
            "suggestions": suggestions_map.get(pred, []),
            "timestamp": datetime.utcnow().isoformat()
        }
    except Exception as e:
        print(f"[ML] classify_portfolio error: {e}")
        return {"type": "balanced", "suggestions": [], "fallback": True}


def analyze_sentiment(headline: str) -> dict:
    """Analyze financial sentiment using FinBERT."""
    try:
        if _finbert_model is None:
            return {"sentiment": "neutral", "confidence": 0.5, "impact_score": 50, "fallback": True}

        inputs = _finbert_tokenizer(headline, return_tensors="pt", truncation=True, max_length=512)
        with torch.no_grad():
            outputs = _finbert_model(**inputs)
            probs = torch.softmax(outputs.logits, dim=1)
            pred = torch.argmax(probs, dim=1).item()
            confidence = probs[0][pred].item()

        labels = ["negative", "neutral", "positive"]
        sentiment = labels[pred]
        impact_score = int(confidence * 100)

        return {
            "sentiment": sentiment,
            "confidence": round(float(confidence), 4),
            "impact_score": impact_score,
            "timestamp": datetime.utcnow().isoformat()
        }
    except Exception as e:
        print(f"[ML] analyze_sentiment error: {e}")
        return {"sentiment": "neutral", "confidence": 0.5, "impact_score": 50, "fallback": True}
'''

with open('ml_loader.py', 'w') as f:
    f.write(ml_loader_code)

print('✅ ml_loader.py written — copy to backend/services/ml_service.py')

In [ ]:
print('✅ All models trained and saved!')
print()
print('📊 Model Summary:')
print(f"   Direction Predictor  : {best_model_name} | Accuracy: {best_accuracy:.4f} | F1: {best_f1:.4f}")
print(f"   Risk Scorer          : RandomForest      | Accuracy: {models_summary['risk_scorer']['accuracy']:.4f}")
print(f"   Portfolio Classifier : RandomForest      | Accuracy: {models_summary['portfolio_classifier']['accuracy']:.4f}")
print(f"   FinBERT              : ProsusAI/finbert  | Loaded from HuggingFace")
print()
print('📁 Models saved to : models/')
print('📄 Summary saved to: models/models_summary.json')
print('🔧 FastAPI helper  : ml_loader.py  →  copy to backend/services/ml_service.py')